# Colab Setup

Mount Google Drive and pull the latest repo changes before running the experiment cells.

In [44]:
!pip install tensordict
!pip install torchRL

In [45]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted


In [46]:
import subprocess
from pathlib import Path

REPO_PATH = Path('/content/drive/MyDrive/repos/evolutionary-ai-battle')

if REPO_PATH.exists():
    result = subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_PATH, text=True, capture_output=True)

    result = subprocess.run(['git', 'pull'], cwd=REPO_PATH, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print(f'Repo path not found: {REPO_PATH}')

Updating 50f311e..7065517
Fast-forward
 experiment/README.md                               |   11 +-
 experiment/check_pr3_acceptance.py                 |    2 +
 experiment/checkpointing.py                        |    4 +
 experiment/configs/ppo_example_tiny.yaml           |    2 +-
 experiment/configs/ppo_smoke.yaml                  |    2 +-
 .../test_reward_selected_checkpoint_and_agents.py  |   27 +-
 experiment/training.ipynb                          | 1253 +++++++-------------
 experiment/training/README.md                      |   18 +-
 experiment/training/train_ppo.py                   |   17 +-
 9 files changed, 462 insertions(+), 874 deletions(-)



In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = REPO_PATH if 'REPO_PATH' in globals() and REPO_PATH.exists() else Path.cwd()
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiment' if PROJECT_ROOT.name != 'experiment' else PROJECT_ROOT

for path in (PROJECT_ROOT, EXPERIMENT_ROOT):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
    print('autoreload enabled')
except Exception as exc:
    print('autoreload skipped:', exc)

try:
    import json
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('DEVICE:', DEVICE)
    if DEVICE == 'cuda':
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA version:', torch.version.cuda)
    else:
        print('No GPU is visible to this runtime. In Colab, use Runtime > Change runtime type > GPU, then rerun setup.')
except Exception as exc:
    DEVICE = 'cpu'
    print('torch device check skipped:', exc)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXPERIMENT_ROOT:', EXPERIMENT_ROOT)

# Notebook training knobs. Adjust these in Colab before running the PPO cell.
CONFIG_PATH = EXPERIMENT_ROOT / 'configs' / 'ppo_smoke.yaml'
RUN_DIR = EXPERIMENT_ROOT / 'runs'
NOTEBOOK_SMOKE = True
NOTEBOOK_TOTAL_STEPS = 256
NOTEBOOK_ROLLOUT_STEPS = 64
NOTEBOOK_MAX_EPISODE_STEPS = 50
NOTEBOOK_HIDDEN_DIM = 64
NOTEBOOK_SELECTION_EVAL_EPISODES = 2
print('CONFIG_PATH:', CONFIG_PATH)
print('RUN_DIR:', RUN_DIR)

# CPC TorchRL Scaffold

This notebook is the initiating experiment entry point. Core logic lives in Python modules; the notebook demonstrates the toy CPC loop, the TorchRL wrapper, and PPO smoke training.

In [47]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()
EXPERIMENT_ROOT = EXPERIMENT_ROOT if 'EXPERIMENT_ROOT' in globals() else PROJECT_ROOT / "experiment"

for path in (PROJECT_ROOT, EXPERIMENT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from training.cpc_actions import action_space, decode_action, describe_action
from training.cpc_env import CPCEnv

print("experiment root:", EXPERIMENT_ROOT)
print("action space:", action_space())

experiment root: /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment
action space: ActionSpace(move_bins=9, aim_bins=16, fire_bins=2)


## PR1: Toy CPC Loop

In [48]:
env = CPCEnv(seed=7, max_steps=8)
obs = env.reset()
move_and_fire = {"move": 6, "aim": 0, "fire": 1}

print(json.dumps(describe_action(move_and_fire), indent=2))
obs, reward, done, info = env.step(move_and_fire)
print("reward", reward, "done", done)
print("decoded", info["decoded_action"])

for _ in range(3):
    action = env.sample_action()
    obs, reward, done, info = env.step(action)
    print({"raw": action, "reward": round(reward, 3), "done": done})
    if done:
        break

print("metrics", json.dumps(env.metrics.summary(), indent=2))
print("trajectory step", json.dumps(env.export_trajectory()["steps"][0], indent=2))

{
  "rawAction": {
    "move": 6,
    "aim": 0,
    "fire": 1
  },
  "moveLabel": "up_right",
  "decoded": {
    "moveX": 0.7071067811865475,
    "moveY": -0.7071067811865475,
    "aimX": 1.0,
    "aimY": 0.0,
    "fire": 1
  }
}
reward 0.02 done False
decoded {'moveX': 0.7071067811865475, 'moveY': -0.7071067811865475, 'aimX': 1.0, 'aimY': 0.0, 'fire': 1}
{'raw': {'move': 5, 'aim': 4, 'fire': 1}, 'reward': 0.02, 'done': False}
{'raw': {'move': 0, 'aim': 2, 'fire': 0}, 'reward': 0.02, 'done': False}
{'raw': {'move': 5, 'aim': 1, 'fire': 0}, 'reward': 0.02, 'done': False}
metrics {
  "avg_ally_distance": 162.45319070603566,
  "isolation_rate": 0.0,
  "teammate_under_pressure_response": 0.0,
  "damage_dealt": 0.0,
  "damage_taken": 0.0
}
trajectory step {
  "step": 1,
  "actorId": "self",
  "action": {
    "moveX": 0.7071067811865475,
    "moveY": -0.7071067811865475,
    "aimX": 1.0,
    "aimY": 0.0,
    "fire": 1
  },
  "rawAction": {
    "move": 6,
    "aim": 0,
    "fire": 1
  },
  "s

## PR2: TorchRL EnvBase / TensorDict Wrapper

In [49]:
try:
    import torch
    from training.torchrl_env import TorchRLCPCEnv
    from training.torchrl_specs import import_check_env_specs

    torchrl_env = TorchRLCPCEnv(seed=11, max_steps=8, device=DEVICE)
    print("observation_spec")
    print(torchrl_env.observation_spec)
    print("action_spec")
    print(torchrl_env.action_spec)

    td = torchrl_env.reset()
    td.update(torchrl_env.action_spec.rand())
    stepped = torchrl_env.step(td)
    print("sample reward", stepped["next", "reward"], "done", stepped["next", "done"])

    move_fire_td = torchrl_env.reset()
    move_fire_td["move"] = torch.tensor(6, dtype=torch.int64)
    move_fire_td["aim"] = torch.tensor(0, dtype=torch.int64)
    move_fire_td["fire"] = torch.tensor(1, dtype=torch.int64)
    move_fire_next = torchrl_env.step(move_fire_td)
    print("decoded engine action", move_fire_next["next", "decoded_action"])

    check_env_specs = import_check_env_specs()
    if check_env_specs is not None:
        check_env_specs(torchrl_env)
        print("check_env_specs passed")
except Exception as exc:
    print("TorchRL wrapper demo skipped:", exc)

observation_spec
Composite(
    self_hp: UnboundedContinuous(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True)),
        device=cuda:0,
        dtype=torch.float32,
        domain=continuous),
    ally_hp: UnboundedContinuous(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True)),
        device=cuda:0,
        dtype=torch.float32,
        domain=continuous),
    enemy_hp: UnboundedContinuous(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True),
            high=Tensor(s

# PR3: PPO Smoke Training

This section trains through the notebook, saves reward-selected checkpoints, and records `checkpoint_selected.pt`. Adjust the notebook training knobs in the setup cell before running this cell.

In [50]:
import json
import shlex
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()
EXPERIMENT_ROOT = EXPERIMENT_ROOT if 'EXPERIMENT_ROOT' in globals() else PROJECT_ROOT / 'experiment'
CONFIG_PATH = CONFIG_PATH if 'CONFIG_PATH' in globals() else EXPERIMENT_ROOT / 'configs' / 'ppo_smoke.yaml'

cmd = [
    sys.executable,
    str(EXPERIMENT_ROOT / 'train_ppo.py'),
    '--config',
    str(CONFIG_PATH),
    '--progress',
]

print('running:')
print(' '.join(shlex.quote(str(part)) for part in cmd))

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

lines = []
for line in process.stdout:
    print(line, end='')
    lines.append(line)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'train_ppo.py failed with exit code {return_code}')

output_text = ''.join(lines)
final_json_start = output_text.rfind('\n{')
if final_json_start == -1:
    final_json_start = output_text.find('{')
else:
    final_json_start += 1
LAST_TRAIN_RESULT = json.loads(output_text[final_json_start:])
SELECTED_CHECKPOINT = LAST_TRAIN_RESULT.get('checkpoint_selected', LAST_TRAIN_RESULT.get('checkpoint_max_reward'))
LATEST_CHECKPOINT = LAST_TRAIN_RESULT['checkpoint_latest']

print('checkpoint_latest:', LATEST_CHECKPOINT)
print('checkpoint_selected:', SELECTED_CHECKPOINT)
print('checkpoint_max_reward:', LAST_TRAIN_RESULT.get('checkpoint_max_reward'))

running:
/usr/bin/python3 /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment/train_ppo.py --config /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment/configs/ppo_smoke.yaml --progress
{"update": 1, "step": 1024, "total_steps": 100000, "episodic_return_mean": 1.978894239407964, "eval_mean_episode_reward": -8.158124502375722, "selection_value": -8.158124502375722, "is_selected_checkpoint": true}
{"update": 2, "step": 2048, "total_steps": 100000, "episodic_return_mean": 4.945106627626552, "eval_mean_episode_reward": 4.60943012535572, "selection_value": 4.60943012535572, "is_selected_checkpoint": true}
{"update": 3, "step": 3072, "total_steps": 100000, "episodic_return_mean": 5.579544978608427, "eval_mean_episode_reward": -14.246841290593148, "selection_value": -14.246841290593148, "is_selected_checkpoint": false}
{"update": 4, "step": 4096, "total_steps": 100000, "episodic_return_mean": 9.20513253159821, "eval_mean_episode_reward": -2.571105930581689, "selection_va

KeyboardInterrupt: 

# PR3 Acceptance Check

This runs the merge-gate checks for PPO smoke training correctness. It checks seed reproducibility, move+fire, raw/decoded action visibility, checkpoint loading, metrics CSV rows, and 10-episode eval robustness.

In [51]:
try:
    from check_pr3_acceptance import check_forced_move_fire_action, run_acceptance

    status, forced = check_forced_move_fire_action(seed=123)
    print("forced action check:", status)
    print(json.dumps(forced, indent=2))

    report = run_acceptance(
        config=EXPERIMENT_ROOT / "configs" / "ppo_smoke.yaml",
        seed=123,
        eval_episodes=10,
        device=DEVICE,
    )
    print(json.dumps(report, indent=2))
except Exception as exc:
    print("PR3 acceptance check skipped:", exc)

forced action check: PASS
{
  "raw_action": {
    "move": 5,
    "aim": 10,
    "fire": 1
  },
  "decoded_action": {
    "move_x": -0.7071067811865475,
    "move_y": -0.7071067811865475,
    "aim_x": -0.7071067811865477,
    "aim_y": -0.7071067811865475,
    "fire": 1
  }
}
{
  "status": "PASS",
  "checks": {
    "same_seed_reproducibility": "PASS",
    "move_fire_step": "PASS",
    "raw_and_decoded_action_visible": "PASS",
    "decode_bounds": "PASS",
    "checkpoint_load_eval": "PASS",
    "metrics_csv_rows": "PASS",
    "eval_10_episodes": "PASS"
  },
  "artifacts": {
    "run_dir": "experiment/runs/ppo_smoke_20260622_110216",
    "checkpoint": "experiment/runs/ppo_smoke_20260622_110216/checkpoint.pt",
    "checkpoint_latest": "experiment/runs/ppo_smoke_20260622_110216/checkpoint_latest.pt",
    "checkpoint_selected": "experiment/runs/ppo_smoke_20260622_110216/checkpoint.pt",
    "checkpoint_min_reward": "experiment/runs/ppo_smoke_20260622_110216/checkpoint_min_reward.pt",
    "chec